[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Dunder Methods &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

Each task builds on the one before it, so the class is written out again each time with the new
method added.


**1.** A repr that reads like the call that would rebuild it.


In [1]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.pages!r})"


print(Book("Dune", "Herbert", 412))
print([Book("Dune", "Herbert", 412), Book("Emma", "Austen", 474)])


Book('Dune', 'Herbert', 412)
[Book('Dune', 'Herbert', 412), Book('Emma', 'Austen', 474)]


`!r` on `pages` makes no visible difference, because the repr of an integer is the same as its str.
It is written anyway so that every field is treated the same way, which matters if `pages` ever
becomes something else.


**2.** Adding the friendly form.


In [2]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.pages!r})"

    def __str__(self):
        return f"{self.title} by {self.author}"


book = Book("Dune", "Herbert", 412)

print("print: ", book)
print("repr:  ", repr(book))
print("a list:", [book])


print:  Dune by Herbert
repr:   Book('Dune', 'Herbert', 412)
a list: [Book('Dune', 'Herbert', 412)]


Three lines, two different strings. The list shows the repr, because containers always do.


**3.** Equality on two fields out of three.


In [3]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.pages!r})"

    def __eq__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return (self.title, self.author) == (other.title, other.author)


first = Book("Dune", "Herbert", 412)
reprint = Book("Dune", "Herbert", 528)
different = Book("Emma", "Austen", 474)

print("same book, different edition:", first == reprint)
print("different book:              ", first == different)
print("against a string:            ", first == "Dune")


same book, different edition: True
different book:               False
against a string:             False


Leaving `pages` out of `__eq__` is the decision this task was about. Two printings of one novel are
the same book, and the page count is a property of the edition.

That is a choice about meaning, not about code, and it is the reason Python does not guess for you.


**4.** The set, before and after `__hash__`.


In [4]:
{first, reprint}


TypeError: cannot use 'Book' as a set element (unhashable type: 'Book')

In [5]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.pages!r})"

    def __eq__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return (self.title, self.author) == (other.title, other.author)

    def __hash__(self):
        return hash((self.title, self.author))


first = Book("Dune", "Herbert", 412)
reprint = Book("Dune", "Herbert", 528)
different = Book("Emma", "Austen", 474)

library = {first, reprint, different}

print("three books in, this many out:", len(library))
print(library)


three books in, this many out: 2
{Book('Dune', 'Herbert', 412), Book('Emma', 'Austen', 474)}


`__hash__` covers the same two fields as `__eq__`, which is the rule. Hashing `pages` as well would
give the two editions different hashes while `__eq__` still called them equal, and the set would
then hold both, which is the inconsistency Python removed the default hash to prevent.

Which of the two survives in the set is whichever went in first, so a set of equal-but-not-identical
objects keeps one arbitrarily.


**5.** `__len__`, and whether it belongs here.


In [6]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.pages!r})"

    def __len__(self):
        return self.pages


print("len:  ", len(Book("Dune", "Herbert", 412)))
print("bool: ", bool(Book("Dune", "Herbert", 412)))
print("empty:", bool(Book("Untitled", "Nobody", 0)))

# __len__ is a poor choice here. It makes a nought-page book falsy, so
# `if book:` quietly means "if it has pages", which is not a question anybody
# asks about a book. A plain `page_count` attribute says the same thing and
# changes nothing else. __len__ belongs on a class that is a collection.


len:   412
bool:  True
empty: False


The comment is the answer. The code runs, and running is not the test.

`__len__` is right for a class that holds a number of things, such as a `Shelf` of books. A `Book`
is one thing, and its page count is an attribute rather than its length.


**6.** Sorting by page count.


In [7]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.pages!r})"

    def __lt__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return self.pages < other.pages


shelf = [Book("Dune", "Herbert", 412), Book("Emma", "Austen", 474),
         Book("Ethan Frome", "Wharton", 99)]

print("sorted:  ", sorted(shelf))
print("shortest:", min(shelf))


sorted:   [Book('Ethan Frome', 'Wharton', 99), Book('Dune', 'Herbert', 412), Book('Emma', 'Austen', 474)]
shortest: Book('Ethan Frome', 'Wharton', 99)


One method, and `sorted`, `min` and `max` all work, because each of them only ever asks which of two
items is smaller.

Page count is a defensible default ordering for books, but title would be just as defensible. When
two orderings are equally natural, leaving `__lt__` out and passing `key=` at each call site says
which one was meant.


---

&#8592; **Back to:** [Dunder Methods](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/04-dunder-methods.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
